In [16]:
import sys, torch
print('python_executable:', sys.executable)
print('torch_version:', torch.__version__)
print('torch_cuda_runtime:', torch.version.cuda)
print('cuda_available:', torch.cuda.is_available())
print('gpu_count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu_name:', torch.cuda.get_device_name(0))

python_executable: c:\Users\gianf\AppData\Local\Programs\Python\Python313\python.exe
torch_version: 2.11.0+cu128
torch_cuda_runtime: 12.8
cuda_available: True
gpu_count: 1
gpu_name: NVIDIA GeForce RTX 3080


Imports

In [17]:
import pandas as pd
import numpy as np
import torch
from umap import UMAP
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

Load Data

In [18]:
df = pd.read_csv('Data/finnhub_model_input.csv')
print('Loaded: Data/finnhub_model_input.csv')
print('Shape:', df.shape)
display(df.head())

Loaded: Data/finnhub_model_input.csv
Shape: (5006, 5)


,stock,date,date_only,headline,headline_raw
0,ING,2024-09-17 15:00:55,2024-09-17,What are share buybacks?,What are share buybacks?
1,SMFG,2025-04-02 03:26:24,2025-04-02,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
2,BACHY,2025-04-02 03:26:24,2025-04-02,PBOC Seen Deploying Stimulus Soon on Tariff Ri...,PBOC Seen Deploying Stimulus Soon on Tariff Ri...
3,BACHY,2025-04-02 07:33:46,2025-04-02,"Across Chinese Fintech And Bank Stocks, See Wh...","Across Chinese Fintech And Bank Stocks, See Wh..."
4,SMFG,2025-04-02 12:41:01,2025-04-02,Japan's __COMPANY__ Plans Stablecoin Developme...,Japan's SMBC Plans Stablecoin Development With...


In [19]:
# 1. Parse date and prepare chronological data
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date', 'headline']).sort_values('date').reset_index(drop=True)

# Deduplicate semantically identical headlines (topic modeling should not overcount copies across tickers)
before_dedup = len(df)
df = df.drop_duplicates(subset=['headline']).sort_values('date').reset_index(drop=True)
after_dedup = len(df)

# 2. Define split proportions and embargo
SPLIT_TRAIN = 0.70
SPLIT_VAL = 0.25
SPLIT_TEST = 0.05
EMBARGO_N = 1  # Number of day-steps dropped between splits to reduce leakage

# 3. Calculate date-based boundaries (keep same day together)
unique_dates = np.array(sorted(df['date'].dt.floor('D').unique()))
n_dates = len(unique_dates)

train_end_idx = int(SPLIT_TRAIN * n_dates)
val_end_idx = int((SPLIT_TRAIN + SPLIT_VAL) * n_dates)

# 4. Create masks with embargo
day_code, _ = pd.factorize(df['date'].dt.floor('D'), sort=True)
mask_train = day_code < train_end_idx
mask_val = (day_code >= (train_end_idx + EMBARGO_N)) & (day_code < val_end_idx)
mask_test = day_code >= (val_end_idx + EMBARGO_N)

# 5. Extract splits
train_df = df[mask_train].copy()
val_df = df[mask_val].copy()
test_df = df[mask_test].copy()

# Final lists for the model
train_docs, train_timestamps = train_df['headline'].tolist(), train_df['date'].tolist()
val_docs, val_timestamps = val_df['headline'].tolist(), val_df['date'].tolist()
test_docs, test_timestamps = test_df['headline'].tolist(), test_df['date'].tolist()

print(f'Deduplication: {before_dedup} -> {after_dedup} rows')
print(f"Train size: {len(train_docs)}")
print(f"Validation size: {len(val_docs)} (Dropped {EMBARGO_N} day units for embargo)")
print(f"Test size: {len(test_docs)} (Dropped {EMBARGO_N} day units for embargo)")
print('Date ranges:')
print(f"  Train: {train_df['date'].min()} -> {train_df['date'].max()}")
print(f"  Val:   {val_df['date'].min()} -> {val_df['date'].max()}")
print(f"  Test:  {test_df['date'].min()} -> {test_df['date'].max()}")

Deduplication: 5006 -> 4300 rows
Train size: 1967
Validation size: 1170 (Dropped 1 day units for embargo)
Test size: 1122 (Dropped 1 day units for embargo)
Date ranges:
  Train: 2024-09-17 15:00:55 -> 2025-12-11 23:21:43
  Val:   2025-12-13 08:05:00 -> 2026-03-10 23:14:51
  Test:  2026-03-12 00:17:03 -> 2026-03-28 19:00:00


Prepare Models

In [20]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Torch CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Add placeholder tokens to stopwords so they do not dominate topics
placeholder_stopwords = {'__ticker__', '__company__', 'ticker', 'company'}
model_stopwords = sorted(set(ENGLISH_STOP_WORDS).union(placeholder_stopwords))

sentence_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
train_embeddings = sentence_model.encode(
    train_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
print(f'Embeddings computed on device: {device}')

Torch CUDA available: True
GPU: NVIDIA GeForce RTX 3080


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1080.27it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1080.27it/s, Materializing param=pooler.dense.weight]                
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 31/31 [00:00<00:00, 51.91it/s]

Embeddings computed on device: cuda


Train BERTopic

In [21]:
print("Training BERTopic model on training split...\n")

umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words=model_stopwords)

topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  calculate_probabilities=True,
  verbose=True
)

# Train BERTopic only on train split
topics_train, probs_train = topic_model.fit_transform(train_docs, embeddings=train_embeddings)

print("\nModel training complete!")

2026-03-29 04:21:41,665 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Training BERTopic model on training split...



2026-03-29 04:21:44,829 - BERTopic - Dimensionality - Completed ✓
2026-03-29 04:21:44,830 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 04:21:44,830 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 04:21:45,054 - BERTopic - Cluster - Completed ✓
2026-03-29 04:21:45,056 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 04:21:45,054 - BERTopic - Cluster - Completed ✓
2026-03-29 04:21:45,056 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 04:21:45,093 - BERTopic - Representation - Completed ✓
2026-03-29 04:21:45,093 - BERTopic - Representation - Completed ✓



Model training complete!


Get Topic Information

In [22]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,669,-1_bank_2025_earnings_week,"[bank, 2025, earnings, week, strong, says, bil...",[__TICKER__ 's Hong Kong Shares Dip After $13....
1,0,82,0_commentary_fund_q1_international,"[commentary, fund, q1, international, 2025, q2...",[Thornburg Better World International Fund Q1 ...
2,1,76,1_china_chinese_gold_stimulus,"[china, chinese, gold, stimulus, pboc, trade, ...","[Wall Street Sees Higher China Growth, Less St..."
3,2,66,2_valuation_recent_assessing_share,"[valuation, recent, assessing, share, dbk, pri...",[Banco __COMPANY__ (BME: __TICKER__ ): Assessi...
4,3,59,3_dividend_portfolio_yield_10,"[dividend, portfolio, yield, 10, stocks, high,...",[My Dividend Stock Portfolio: New July Dividen...
5,4,51,4_stablecoin_crypto_banks_major,"[stablecoin, crypto, banks, major, swift, bloc...",[Japan’s Big Stablecoin Play: 3 Major Banks to...
6,5,48,5_loan_energy_financing_secures,"[loan, energy, financing, secures, debt, billi...",['Banks Ready $38 Billion of Debt for Data Cen...
7,6,47,6_afternoon_update_sector_financial,"[afternoon, update, sector, financial, stocks,...",[Sector Update: Financial Stocks Rise Late Aft...
8,7,46,7_buyback_share_programme_voting,"[buyback, share, programme, voting, rights, sh...",[__COMPANY__ share buyback programme - Declara...
9,8,33,8_ai_technology_driven_innovation,"[ai, technology, driven, innovation, joins, ge...",[__TICKER__ joins the Massachusetts Institute ...


Visualize Topics

In [23]:
topic_model.visualize_documents(train_docs, embeddings=train_embeddings)

In [24]:
fig = topic_model.visualize_heatmap()
fig.show()

In [25]:
hierarchical_topics = topic_model.hierarchical_topics(train_docs)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

print("💡 This shows the hierarchical structure of topics")

100%|██████████| 49/49 [00:00<00:00, 609.79it/s]



💡 This shows the hierarchical structure of topics


In [26]:
import plotly.express as px

topics_over_time = topic_model.topics_over_time(
    train_docs,
    train_timestamps,
    nr_bins=20
 )

tot = topics_over_time.copy()
tot = tot[tot['Topic'] != -1].copy()
tot['Timestamp'] = pd.to_datetime(tot['Timestamp']).dt.to_period('M').dt.to_timestamp()

topic_labels = topic_model.get_topic_info()[['Topic', 'Name']].copy()
topic_labels = topic_labels[topic_labels['Topic'] != -1]
tot = tot.merge(topic_labels, on='Topic', how='left')
tot['TopicLabel'] = tot['Topic'].astype(str) + ': ' + tot['Name'].fillna('')

stacked = (
    tot.groupby(['Timestamp', 'TopicLabel'], as_index=False)['Frequency']
       .sum()
       .sort_values(['Timestamp', 'Frequency'], ascending=[True, False])
)

fig = px.bar(
    stacked,
    x='Timestamp',
    y='Frequency',
    color='TopicLabel',
    title='Topic Frequency Over Time (Stacked)',
    labels={'Timestamp': 'Month', 'Frequency': 'Document Count', 'TopicLabel': 'Topic'}
)
fig.update_layout(
    barmode='stack',
    xaxis_title='Month',
    yaxis_title='Document Count',
    legend_title='Topic',
    hovermode='x unified'
 )
fig.show()

13it [00:00, 50.29it/s]



Preparing Validation

In [27]:
bertopic_topics = [
    [word for word, _ in topic_model.get_topic(t)]
    for t in topic_model.get_topics().keys() if t != -1
    if topic_model.get_topic(t) is not None
 ]

# Project validation documents into trained topic space
val_topics, val_probs = topic_model.transform(val_docs)

print("Validation transform complete (test set remains untouched).")

Batches: 100%|██████████| 37/37 [00:00<00:00, 90.17it/s] 
2026-03-29 04:21:49,582 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
Batches: 100%|██████████| 37/37 [00:00<00:00, 90.17it/s] 
2026-03-29 04:21:49,582 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-29 04:21:53,054 - BERTopic - Dimensionality - Completed ✓
2026-03-29 04:21:53,055 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-29 04:21:53,054 - BERTopic - Dimensionality - Completed ✓
2026-03-29 04:21:53,055 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-29 04:21:53,115 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-03-29 04:21:53,115 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-03-29 04:21:53,256 - BERTopic - Probabilities - Completed ✓
2026-03-29 04:21:53,257 - BERTopic - Cluster - Completed ✓
2026-03-29 04:21:53,256 - BERTopic -

Validation transform complete (test set remains untouched).


In [31]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.feature_extraction.text import CountVectorizer
import nltk
import warnings
from nltk.corpus import stopwords

# Download required NLTK data
nltk.download('stopwords', quiet=True)

# Build a tokenizer aligned with BERTopic vocabulary (supports unigrams + bigrams)
stop_words = list(model_stopwords) if 'model_stopwords' in globals() else list(stopwords.words('english'))
coherence_vectorizer = CountVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b'
)
coherence_analyzer = coherence_vectorizer.build_analyzer()

def preprocess_documents(doc_list):
    tokenized = [coherence_analyzer(str(doc).lower()) for doc in doc_list]
    return [tokens for tokens in tokenized if len(tokens) >= 2]

processed_train_docs = preprocess_documents(train_docs)
processed_val_docs = preprocess_documents(val_docs)

# Dictionary from train split only
id2word = Dictionary(processed_train_docs)

def topic_diversity(topics):
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    return len(unique_words) / len(all_words) if len(all_words) > 0 else 0.0

def _filter_topics_for_dictionary(topics, dictionary, min_words_per_topic=3):
    vocab = dictionary.token2id
    filtered = []
    for topic in topics:
        cleaned = []
        for term in topic:
            token = str(term).strip().lower()
            if token in vocab:
                cleaned.append(token)
        cleaned = list(dict.fromkeys(cleaned))
        if len(cleaned) >= min_words_per_topic:
            filtered.append(cleaned)
    return filtered

def coherence_score(topics, texts, dictionary, coherence_type='c_v'):
    clean_texts = [t for t in texts if isinstance(t, list) and len(t) >= 2]
    if len(clean_texts) < 20 or len(dictionary) < 30:
        return np.nan

    local_dict = Dictionary(clean_texts)
    local_dict.filter_extremes(no_below=2, no_above=0.95)
    local_dict.compactify()
    if len(local_dict) < 20:
        return np.nan

    filtered_topics = _filter_topics_for_dictionary(topics, local_dict, min_words_per_topic=3)
    if len(filtered_topics) < 2:
        return np.nan

    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=RuntimeWarning, module='gensim')
            with np.errstate(divide='ignore', invalid='ignore'):
                cm = CoherenceModel(
                    topics=filtered_topics,
                    texts=clean_texts,
                    dictionary=local_dict,
                    coherence=coherence_type
                )
                value = float(cm.get_coherence())
        return value if np.isfinite(value) else np.nan
    except Exception:
        return np.nan

cv_train = coherence_score(bertopic_topics, processed_train_docs, id2word, 'c_v')
cv_val = coherence_score(bertopic_topics, processed_val_docs, id2word, 'c_v')
div_train = topic_diversity(bertopic_topics)
val_outlier_ratio = float(np.mean(np.array(val_topics) == -1)) if len(val_topics) > 0 else np.nan
valid_topic_count = len(_filter_topics_for_dictionary(bertopic_topics, id2word, min_words_per_topic=3))

print(f"BERTopic Coherence C_v (train): {cv_train:.4f}")
print(f"BERTopic Coherence C_v (val):   {cv_val:.4f}")
print(f"BERTopic Topic Diversity (train): {div_train:.4f}")
print(f"Validation outlier ratio (-1 topics): {val_outlier_ratio:.4f}")
print(f"Valid topic count for coherence: {valid_topic_count}")

AttributeError: 'Dictionary' object has no attribute 'copy'

Hyperparameter tuning with rolling time-series CV (train+val only), multi-seed reproducibility, and bucket-wise reporting

In [ ]:
from itertools import product
import time

# -------------------- Time-series CV + reproducibility controls --------------------
SEEDS = [42, 7]
N_FOLDS = 3  # rolling/expanding buckets before test
EMBARGO_FOLD = 1
FAST_TUNING = True
MAX_TRIALS_FAST = 8
TOPK_FULL_COHERENCE_FAST = 4
RANDOM_SEED = 42

# Build tuning pool from train + validation only (test remains untouched)
tune_df = pd.concat([train_df, val_df], axis=0).sort_values('date').reset_index(drop=True)
tune_docs = tune_df['headline'].tolist()
tune_timestamps = tune_df['date'].tolist()

# Precompute embeddings for the full tuning pool once
tune_embeddings = sentence_model.encode(
    tune_docs,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

def sanitize_metric(value):
    try:
        v = float(value)
    except Exception:
        return np.nan
    return v if np.isfinite(v) else np.nan

def make_expanding_folds(df_dates, n_folds=3, embargo_n=1):
    unique_days = np.array(sorted(pd.to_datetime(df_dates).dt.floor('D').unique()))
    n_days = len(unique_days)
    edges = np.linspace(0, n_days, n_folds + 2, dtype=int)
    day_code, _ = pd.factorize(pd.to_datetime(df_dates).dt.floor('D'), sort=True)

    folds = []
    for fold_idx in range(n_folds):
        train_end = edges[fold_idx + 1]
        val_start = train_end + embargo_n
        val_end = edges[fold_idx + 2]

        if train_end <= 0 or val_start >= val_end:
            continue

        train_mask = day_code < train_end
        val_mask = (day_code >= val_start) & (day_code < val_end)

        train_idx = np.where(train_mask)[0]
        val_idx = np.where(val_mask)[0]

        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        folds.append({
            'fold': fold_idx + 1,
            'train_idx': train_idx,
            'val_idx': val_idx,
            'train_start': unique_days[0],
            'train_end': unique_days[train_end - 1],
            'val_start': unique_days[val_start],
            'val_end': unique_days[val_end - 1]
        })

    return folds

folds = make_expanding_folds(tune_df['date'], n_folds=N_FOLDS, embargo_n=EMBARGO_FOLD)
print(f'Generated {len(folds)} rolling folds (train+val pool only).')
for fold in folds:
    print(
        f"Fold {fold['fold']}: train {fold['train_start']} -> {fold['train_end']} | "
        f"val {fold['val_start']} -> {fold['val_end']}"
    )

param_grid = {
    'n_neighbors': [10, 15],
    'n_components': [5, 10],
    'min_cluster_size': [10, 20],
    'min_samples': [5],
    'ngram_range': [(1, 1), (1, 2)]
}

full_search_space = list(product(
    param_grid['n_neighbors'],
    param_grid['n_components'],
    param_grid['min_cluster_size'],
    param_grid['min_samples'],
    param_grid['ngram_range']
))

rng = np.random.default_rng(RANDOM_SEED)
if FAST_TUNING and len(full_search_space) > MAX_TRIALS_FAST:
    selected_idx = rng.choice(len(full_search_space), size=MAX_TRIALS_FAST, replace=False)
    search_space = [full_search_space[i] for i in selected_idx]
else:
    search_space = full_search_space

print(f'Running {len(search_space)} parameter combinations x {len(folds)} folds x {len(SEEDS)} seeds')

cv_rows = []

for trial, (n_neighbors, n_components, min_cluster_size, min_samples, ngram_range) in enumerate(search_space, start=1):
    print(
        f"[Trial {trial}/{len(search_space)}] n_neighbors={n_neighbors}, n_components={n_components}, "
        f"min_cluster_size={min_cluster_size}, min_samples={min_samples}, ngram_range={ngram_range}"
    )

    for fold in folds:
        train_idx = fold['train_idx']
        val_idx = fold['val_idx']

        fold_train_docs = [tune_docs[i] for i in train_idx]
        fold_val_docs = [tune_docs[i] for i in val_idx]
        fold_train_embeddings = tune_embeddings[train_idx]
        fold_val_embeddings = tune_embeddings[val_idx]

        processed_train_fold = preprocess_documents(fold_train_docs)
        processed_val_fold = preprocess_documents(fold_val_docs)
        id2word_fold = Dictionary(processed_train_fold)

        for seed in SEEDS:
            start = time.perf_counter()

            umap_model_i = UMAP(
                n_neighbors=n_neighbors,
                n_components=n_components,
                metric='cosine',
                random_state=seed
            )
            hdbscan_model_i = HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric='euclidean',
                cluster_selection_method='eom',
                prediction_data=True,
                core_dist_n_jobs=-1
            )
            vectorizer_model_i = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=ngram_range)

            topic_model_i = BERTopic(
                embedding_model=sentence_model,
                umap_model=umap_model_i,
                hdbscan_model=hdbscan_model_i,
                vectorizer_model=vectorizer_model_i,
                calculate_probabilities=False,
                verbose=False
            )

            _topics_train_i, _ = topic_model_i.fit_transform(fold_train_docs, embeddings=fold_train_embeddings)
            val_topics_i, _ = topic_model_i.transform(fold_val_docs, embeddings=fold_val_embeddings)

            topic_words_i = [
                [word for word, _ in topic_model_i.get_topic(t)]
                for t in topic_model_i.get_topics().keys()
                if t != -1 and topic_model_i.get_topic(t) is not None
            ]

            # Coherence on all combinations is expensive and unstable on sparse folds
            has_enough_for_cv = (
                len(processed_val_fold) >= 20 and
                len(id2word_fold) >= 30 and
                len(topic_words_i) >= 2
            )
            do_cv = (
                trial <= TOPK_FULL_COHERENCE_FAST and
                fold['fold'] <= 2 and
                seed == SEEDS[0] and
                has_enough_for_cv
            )
            cv_val_i = sanitize_metric(
                coherence_score(topic_words_i, processed_val_fold, id2word_fold, 'c_v')
            ) if do_cv else np.nan

            div_i = sanitize_metric(topic_diversity(topic_words_i))
            outlier_i = sanitize_metric(float(np.mean(np.array(val_topics_i) == -1)) if len(val_topics_i) > 0 else np.nan)
            n_topics_i = int(sum(1 for t in topic_model_i.get_topics().keys() if int(t) != -1))

            elapsed = time.perf_counter() - start

            cv_rows.append({
                'trial': trial,
                'fold': fold['fold'],
                'seed': seed,
                'n_neighbors': n_neighbors,
                'n_components': n_components,
                'min_cluster_size': min_cluster_size,
                'min_samples': min_samples,
                'ngram_range': str(ngram_range),
                'n_topics': n_topics_i,
                'cv_val': cv_val_i,
                'topic_diversity': div_i,
                'val_outlier_ratio': outlier_i,
                'fit_seconds': elapsed
            })

cv_results = pd.DataFrame(cv_rows)

agg_cols = ['cv_val', 'topic_diversity', 'val_outlier_ratio', 'n_topics', 'fit_seconds']
summary = cv_results.groupby(
    ['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
 )[agg_cols].mean()

# Reproducibility / stability: std across seed-fold realizations
stability = cv_results.groupby(
    ['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
 )[['cv_val', 'topic_diversity', 'val_outlier_ratio']].std()
stability = stability.rename(columns={
    'cv_val': 'cv_val_std',
    'topic_diversity': 'topic_diversity_std',
    'val_outlier_ratio': 'val_outlier_ratio_std'
})

tuning_results = summary.merge(
    stability,
    on=['n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    how='left'
 )

for col in ['cv_val_std', 'topic_diversity_std', 'val_outlier_ratio_std']:
    tuning_results[col] = tuning_results[col].fillna(0.0)

def minmax_norm(series, higher_is_better=True):
    s = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    if s.dropna().empty:
        return pd.Series(0.0, index=s.index)
    filled = s.fillna(s.min() if higher_is_better else s.max())
    min_v, max_v = float(filled.min()), float(filled.max())
    if max_v == min_v:
        return pd.Series(0.0, index=filled.index)
    norm = (filled - min_v) / (max_v - min_v)
    return norm

tuning_results['cv_val_norm'] = minmax_norm(tuning_results['cv_val'], higher_is_better=True)
tuning_results['topic_diversity_norm'] = minmax_norm(tuning_results['topic_diversity'], higher_is_better=True)
tuning_results['val_outlier_ratio_norm'] = minmax_norm(tuning_results['val_outlier_ratio'], higher_is_better=False)
tuning_results['cv_val_std_norm'] = minmax_norm(tuning_results['cv_val_std'], higher_is_better=False)

# Penalize too few topics
tuning_results['topic_count_penalty'] = np.where(tuning_results['n_topics'] < 3, 0.25, 0.0)

# Composite: quality + stability
tuning_results['composite_score'] = (
    0.45 * tuning_results['cv_val_norm'] +
    0.25 * tuning_results['topic_diversity_norm'] +
    0.15 * tuning_results['val_outlier_ratio_norm'] +
    0.15 * tuning_results['cv_val_std_norm'] -
    tuning_results['topic_count_penalty']
)

tuning_results = tuning_results.sort_values('composite_score', ascending=False).reset_index(drop=True)
best_params = tuning_results.iloc[0].to_dict()

# Bucket-wise (fold-wise) best params to inspect drift over time
fold_agg = cv_results.groupby(
    ['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range'],
    as_index=False
 )[['cv_val', 'topic_diversity', 'val_outlier_ratio', 'n_topics']].mean()

fold_agg['cv_val_norm'] = fold_agg.groupby('fold')['cv_val'].transform(lambda x: minmax_norm(x, True))
fold_agg['topic_diversity_norm'] = fold_agg.groupby('fold')['topic_diversity'].transform(lambda x: minmax_norm(x, True))
fold_agg['val_outlier_ratio_norm'] = fold_agg.groupby('fold')['val_outlier_ratio'].transform(lambda x: minmax_norm(x, False))
fold_agg['topic_count_penalty'] = np.where(fold_agg['n_topics'] < 3, 0.25, 0.0)
fold_agg['fold_score'] = (
    0.50 * fold_agg['cv_val_norm'] +
    0.30 * fold_agg['topic_diversity_norm'] +
    0.20 * fold_agg['val_outlier_ratio_norm'] -
    fold_agg['topic_count_penalty']
)

best_per_fold = (
    fold_agg.sort_values(['fold', 'fold_score'], ascending=[True, False])
            .groupby('fold', as_index=False)
            .head(1)
            .reset_index(drop=True)
 )

print('\nTop configurations (overall):')
display(tuning_results.head(10))

print('\nBest params by time bucket (fold):')
display(best_per_fold[['fold', 'n_neighbors', 'n_components', 'min_cluster_size', 'min_samples', 'ngram_range', 'fold_score']])

print('\nAverage fit time per model run (seconds):', round(float(cv_results['fit_seconds'].mean()), 2))
print('Best overall configuration selected from rolling CV + multi-seed composite score:')
display(pd.DataFrame([best_params]))

Batches: 100%|██████████| 50/50 [00:00<00:00, 63.46it/s]



Generated 3 rolling folds (train+val pool only).
Fold 1: train 2024-09-17 00:00:00 -> 2025-07-03 00:00:00 | val 2025-07-05 00:00:00 -> 2025-09-23 00:00:00
Fold 2: train 2024-09-17 00:00:00 -> 2025-09-23 00:00:00 | val 2025-09-25 00:00:00 -> 2025-12-15 00:00:00
Fold 3: train 2024-09-17 00:00:00 -> 2025-12-15 00:00:00 | val 2025-12-17 00:00:00 -> 2026-03-10 00:00:00
Running 8 parameter combinations x 3 folds x 2 seeds
[Trial 1/8] n_neighbors=10, n_components=5, min_cluster_size=10, min_samples=5, ngram_range=(1, 2)


C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\indirect_confirmation_measure.py:323: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\indirect_confirmation_measure.py:323: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherenc

[Trial 2/8] n_neighbors=15, n_components=5, min_cluster_size=20, min_samples=5, ngram_range=(1, 2)


C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\indirect_confirmation_measure.py:323: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\indirect_confirmation_measure.py:323: RuntimeWarning:

invalid value encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherence\direct_confirmation_measure.py:204: RuntimeWarning:

divide by zero encountered in scalar divide

C:\Users\gianf\AppData\Roaming\Python\Python313\site-packages\gensim\topic_coherenc

...

,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,topic_diversity,val_outlier_ratio,n_topics,fit_seconds,cv_val_std,topic_diversity_std,val_outlier_ratio_std,cv_val_norm,topic_diversity_norm,val_outlier_ratio_norm,cv_val_std_norm,topic_count_penalty,composite_score
0,15,10,10,5,"(1, 2)",NaN,0.915933,0.571775,35.333333,9.712745,0.0,0.010427,0.095253,0.0,0.659598,1.000000,0.0,0.0,0.314900
1,10,10,20,5,"(1, 2)",NaN,0.924706,0.482369,18.666667,4.543041,0.0,0.011139,0.022740,0.0,0.805765,0.424218,0.0,0.0,0.265074
2,15,5,20,5,"(1, 2)",NaN,0.936364,0.416497,16.666667,9.569753,0.0,0.021144,0.211804,0.0,1.000000,0.000000,0.0,0.0,0.250000
3,10,10,10,5,"(1, 2)",NaN,0.911893,0.496519,37.666667,4.471742,0.0,0.022772,0.078510,0.0,0.592278,0.515345,0.0,0.0,0.225371
4,15,10,10,5,"(1, 1)",NaN,0.886534,0.571775,35.333333,4.701778,0.0,0.034975,0.095253,0.0,0.169773,1.000000,0.0,0.0,0.192443
5,10,5,10,5,"(1, 2)",NaN,0.907714,0.459277,38.000000,9.539496,0.0,0.019136,0.032418,0.0,0.522650,0.275507,0.0,0.0,0.171988
6,15,5,20,5,"(1, 1)",NaN,0.913552,0.416497,16.666667,9.598065,0.0,0.030679,0.211804,0.0,0.619928,0.000000,0.0,0.0,0.154982
7,10,5,10,5,"(1, 1)",NaN,0.876345,0.459277,38.000000,4.506800,0.0,0.042345,0.032418,0.0,0.000000,0.275507,0.0,0.0,0.041326



Best params by time bucket (fold):


,fold,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,fold_score
0,1,10,10,10,5,"(1, 2)",0.382398
1,2,10,10,20,5,"(1, 2)",0.430137
2,3,10,10,20,5,"(1, 2)",0.447933



Average fit time per model run (seconds): 7.08
Best overall configuration selected from rolling CV + multi-seed composite score:


,n_neighbors,n_components,min_cluster_size,min_samples,ngram_range,cv_val,topic_diversity,val_outlier_ratio,n_topics,fit_seconds,cv_val_std,topic_diversity_std,val_outlier_ratio_std,cv_val_norm,topic_diversity_norm,val_outlier_ratio_norm,cv_val_std_norm,topic_count_penalty,composite_score
0,15,10,10,5,"(1, 2)",NaN,0.915933,0.571775,35.333333,9.712745,0.0,0.010427,0.095253,0.0,0.659598,1.0,0.0,0.0,0.3149


Final model and test evaluation (single touch of test split)

In [30]:
from ast import literal_eval

# Refit best model on train+val, evaluate once on test
train_val_docs = train_docs + val_docs
train_val_embeddings = sentence_model.encode(train_val_docs, show_progress_bar=True)
test_embeddings = sentence_model.encode(test_docs, show_progress_bar=True)

best_umap = UMAP(
    n_neighbors=int(best_params['n_neighbors']),
    n_components=int(best_params['n_components']),
    metric='cosine',
    random_state=42
)
best_hdbscan = HDBSCAN(
    min_cluster_size=int(best_params['min_cluster_size']),
    min_samples=int(best_params['min_samples']),
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)
best_ngram = literal_eval(best_params['ngram_range'])
best_vectorizer = OnlineCountVectorizer(stop_words=model_stopwords, ngram_range=best_ngram)

best_topic_model = BERTopic(
    embedding_model=sentence_model,
    umap_model=best_umap,
    hdbscan_model=best_hdbscan,
    vectorizer_model=best_vectorizer,
    calculate_probabilities=True,
    verbose=True
)

_topics_train_val, _ = best_topic_model.fit_transform(train_val_docs, embeddings=train_val_embeddings)
test_topics, _ = best_topic_model.transform(test_docs, embeddings=test_embeddings)

best_topic_words = [
    [word for word, _ in best_topic_model.get_topic(t)]
    for t in best_topic_model.get_topics().keys()
    if t != -1 and best_topic_model.get_topic(t) is not None
]

processed_train_val_docs = preprocess_documents(train_val_docs)
processed_test_docs = preprocess_documents(test_docs)
id2word_train_val = Dictionary(processed_train_val_docs)

cv_test = coherence_score(best_topic_words, processed_test_docs, id2word_train_val, 'c_v')
div_test = topic_diversity(best_topic_words)
test_outlier_ratio = float(np.mean(np.array(test_topics) == -1)) if len(test_topics) > 0 else np.nan
test_topic_count = int(sum(1 for t in best_topic_model.get_topics().keys() if int(t) != -1))

print('Final Test Metrics (best config):')
print(f"C_v coherence (test): {cv_test:.4f}")
print(f"Topic diversity: {div_test:.4f}")
print(f"Test outlier ratio (-1 topics): {test_outlier_ratio:.4f}")
print(f"Topic count (excluding -1): {test_topic_count}")

best_topic_model.get_topic_info().head(15)

Batches: 100%|██████████| 36/36 [00:00<00:00, 139.06it/s]
2026-03-29 04:28:07,337 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
Batches: 100%|██████████| 36/36 [00:00<00:00, 139.06it/s]
2026-03-29 04:28:07,337 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-29 04:28:18,061 - BERTopic - Dimensionality - Completed ✓
2026-03-29 04:28:18,062 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 04:28:18,061 - BERTopic - Dimensionality - Completed ✓
2026-03-29 04:28:18,062 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 04:28:18,620 - BERTopic - Cluster - Completed ✓
2026-03-29 04:28:18,622 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 04:28:18,620 - BERTopic - Cluster - Completed ✓
2026-03-29 04:28:18,622 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 04:28:18,710 - BERTopic - Representation - 

Final Test Metrics (best config):
C_v coherence (test): nan
Topic diversity: 0.8718
Test outlier ratio (-1 topics): 0.5187
Topic count (excluding -1): 85


,Topic,Count,Name,Representation,Representative_Docs
0,-1,958,-1_market_stock_bank_2026,"[market, stock, bank, 2026, 2025, earnings, we...",[Half-year report on Pluxee N.V.’s liquidity c...
1,0,105,0_stablecoin_crypto_banks_blockchain,"[stablecoin, crypto, banks, blockchain, banco,...",[__COMPANY__ ’s Crypto Arm to Launch Dollar De...
2,1,90,1_commentary_fund_2025 commentary_q2 2025,"[commentary, fund, 2025 commentary, q2 2025, f...",[American Century International Growth Fund Q1...
3,2,62,2_ai_technology_ai innovation_driven,"[ai, technology, ai innovation, driven, amazon...",[__TICKER__ Collaborates With Mistral AI to Bo...
4,3,61,3_loan_energy_financing_billion,"[loan, energy, financing, billion, million, de...","[Update: Market Chatter: __COMPANY__ , __COMPA..."
5,4,59,4_sector update_update financial_financial sto...,"[sector update, update financial, financial st...",[Sector Update: Financial Stocks Mixed Late Af...
6,5,52,5_profit_profit rises_rote_growth,"[profit, profit rises, rote, growth, posts, pr...",[__COMPANY__ Targets ~11% RoTE in 2025; Sees >...
7,6,51,6_shifting_narrative_story_targets,"[shifting, narrative, story, targets, analyst,...",[How The Narrative On __COMPANY__ Bank ( __TIC...
8,7,47,7_etf_etfs_spdr_investment management,"[etf, etfs, spdr, investment management, cap e...",[__COMPANY__ Investment Management Unveils New...
9,8,46,8_valuation_recent_share price_recent share,"[valuation, recent, share price, recent share,...",[Assessing __TICKER__ ( __TICKER__ )’s Valuati...
